# Multi-Class Text Classification: A Comparison of Word Representations and ML/NN Models


## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, SimpleRNN, GRU, LSTM, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from wordcloud import WordCloud
from collections import Counter

print("All libraries imported successfully!")

## 2. Load Dataset

In [ ]:
train_df = pd.read_csv('Question Answer Classification Dataset 5[Training].csv')
test_df = pd.read_csv('Question Answer Classification Dataset[Test].csv')

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

In [ ]:
print("\n=== Training Data Sample ===")
train_df.head()

In [ ]:
print("\n=== Testing Data Sample ===")
test_df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("=== Training Dataset Info ===")
print(train_df.info())
print("\n=== Statistical Summary ===")
train_df.describe()

In [ ]:
print("=== Missing Values ===")
print("Training set:")
print(train_df.isnull().sum())
print("\nTesting set:")
print(test_df.isnull().sum())

In [ ]:
print("=== Duplicate Rows ===")
print(f"Training duplicates: {train_df.duplicated().sum()}")
print(f"Testing duplicates: {test_df.duplicated().sum()}")

In [ ]:
print("=== Class Distribution (Training) ===")
class_counts_train = train_df['Class'].value_counts()
print(class_counts_train)
print(f"\nTotal classes: {len(class_counts_train)}")

In [ ]:
plt.figure(figsize=(12, 6))
colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts_train)))
bars = plt.bar(range(len(class_counts_train)), class_counts_train.values, color=colors)
plt.xticks(range(len(class_counts_train)), class_counts_train.index, rotation=45, ha='right')
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution in Training Dataset')

for bar, count in zip(bars, class_counts_train.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
plt.pie(class_counts_train.values, labels=class_counts_train.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
plt.title('Class Distribution (Percentage) - Training Data')
plt.axis('equal')
plt.show()

In [ ]:
train_df['text_length'] = train_df['QA Text'].apply(len)
train_df['word_count'] = train_df['QA Text'].apply(lambda x: len(str(x).split()))

print("=== Text Length Statistics ===")
print(train_df[['text_length', 'word_count']].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['text_length'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Text Length (Characters)')
axes[0].axvline(train_df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["text_length"].mean():.0f}')
axes[0].legend()

axes[1].hist(train_df['word_count'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Word Count')
axes[1].axvline(train_df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["word_count"].mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
train_df.boxplot(column='word_count', by='Class', figsize=(14, 6))
plt.xticks(rotation=45, ha='right')
plt.xlabel('Class')
plt.ylabel('Word Count')
plt.title('Word Count Distribution by Class')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
all_text = ' '.join(train_df['QA Text'].astype(str).tolist())
words = all_text.lower().split()

word_freq = Counter(words)
most_common = word_freq.most_common(30)

print("=== Top 30 Most Frequent Words (Before Preprocessing) ===")
for word, count in most_common:
    print(f"{word}: {count}")

In [ ]:
words_list = [w[0] for w in most_common]
counts_list = [w[1] for w in most_common]

plt.figure(figsize=(14, 6))
plt.barh(range(len(words_list)), counts_list, color='teal')
plt.yticks(range(len(words_list)), words_list)
plt.xlabel('Frequency')
plt.ylabel('Words')
plt.title('Top 30 Most Frequent Words (Before Preprocessing)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
wordcloud = WordCloud(width=1500, height=800, background_color='white',
                      max_words=200, colormap='viridis').generate(all_text)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - All Training Data', fontsize=16)
plt.show()

In [ ]:
classes = train_df['Class'].unique()
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, cls in enumerate(classes):
    class_text = ' '.join(train_df[train_df['Class'] == cls]['QA Text'].astype(str).tolist())
    wordcloud = WordCloud(width=400, height=300, background_color='white',
                          max_words=100, colormap='viridis').generate(class_text)
    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].axis('off')
    axes[idx].set_title(cls, fontsize=10)

plt.suptitle('Word Clouds by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Sample Texts from Each Class ===")
for cls in train_df['Class'].unique():
    print(f"\n--- {cls} ---")
    sample = train_df[train_df['Class'] == cls]['QA Text'].iloc[0]
    print(sample[:300] + "..." if len(sample) > 300 else sample)

## 4. Data Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'question title:', '', text)
    text = re.sub(r'question content:', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join(text.split())
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens
              if token not in stop_words and len(token) > 2]
    return ' '.join(tokens)

print("Text cleaning function defined successfully!")

In [ ]:
sample_text = train_df['QA Text'].iloc[0]
print("Original text:")
print(sample_text[:500])
print("\nCleaned text:")
print(clean_text(sample_text)[:500])

In [ ]:
print("Preprocessing training data...")
train_df['cleaned_text'] = train_df['QA Text'].apply(clean_text)

print("Preprocessing testing data...")
test_df['cleaned_text'] = test_df['QA Text'].apply(clean_text)

print("Preprocessing completed!")

In [ ]:
print("=== Sample Preprocessed Data ===")
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"Original: {train_df['QA Text'].iloc[i][:200]}...")
    print(f"Cleaned: {train_df['cleaned_text'].iloc[i][:200]}...")

In [ ]:
train_df = train_df[train_df['cleaned_text'].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df['cleaned_text'].str.len() > 0].reset_index(drop=True)

print(f"Training samples after cleaning: {len(train_df)}")
print(f"Testing samples after cleaning: {len(test_df)}")

In [ ]:
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['Class'])
test_df['label_encoded'] = label_encoder.transform(test_df['Class'])

num_classes = len(label_encoder.classes_)

print("=== Label Encoding ===")
for i, cls in enumerate(label_encoder.classes_):
    print(f"{i}: {cls}")
print(f"\nTotal number of classes: {num_classes}")

In [ ]:
X_train = train_df['cleaned_text'].values
y_train = train_df['label_encoded'].values

X_test = test_df['cleaned_text'].values
y_test = test_df['label_encoded'].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f"Training set (for training): {len(X_train_split)}")
print(f"Validation set: {len(X_val)}")
print(f"Test set: {len(X_test)}")

## 5. Word Representation Techniques

In [ ]:
bow_vectorizer = CountVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_bow = bow_vectorizer.fit_transform(X_train_split)
X_val_bow = bow_vectorizer.transform(X_val)
X_test_bow = bow_vectorizer.transform(X_test)

print(f"BoW Training shape: {X_train_bow.shape}")
print(f"BoW Validation shape: {X_val_bow.shape}")
print(f"BoW Test shape: {X_test_bow.shape}")
print(f"\nVocabulary size: {len(bow_vectorizer.vocabulary_)}")

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.95)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_split)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF Training shape: {X_train_tfidf.shape}")
print(f"TF-IDF Validation shape: {X_val_tfidf.shape}")
print(f"TF-IDF Test shape: {X_test_tfidf.shape}")
print(f"\nVocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 100

nn_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
nn_tokenizer.fit_on_texts(X_train_split)

X_train_seq = nn_tokenizer.texts_to_sequences(X_train_split)
X_val_seq = nn_tokenizer.texts_to_sequences(X_val)
X_test_seq = nn_tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

vocab_size = min(len(nn_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

print(f"Vocabulary size: {vocab_size}")
print(f"Max sequence length: {MAX_SEQUENCE_LENGTH}")
print(f"\nPadded Training shape: {X_train_pad.shape}")
print(f"Padded Validation shape: {X_val_pad.shape}")
print(f"Padded Test shape: {X_test_pad.shape}")

In [ ]:
y_train_cat = to_categorical(y_train_split, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print(f"y_train_cat shape: {y_train_cat.shape}")
print(f"y_val_cat shape: {y_val_cat.shape}")
print(f"y_test_cat shape: {y_test_cat.shape}")

In [ ]:
def load_glove_embeddings(glove_file_path):
    embeddings_index = {}
    try:
        with open(glove_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                values = line.split()
                word = values[0]
                vector = np.asarray(values[1:], dtype='float32')
                embeddings_index[word] = vector
        print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    except FileNotFoundError:
        print("GloVe file not found. Using random embeddings instead.")
        print("Download glove.6B.100d.txt from https://nlp.stanford.edu/projects/glove/")
    return embeddings_index

GLOVE_PATH = 'glove.6B.100d.txt'
glove_embeddings_index = load_glove_embeddings(GLOVE_PATH)

In [ ]:
def create_embedding_matrix(word_index, embeddings_source, embedding_dim, vocab_size, is_word2vec=False):
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    found_count = 0

    for word, i in word_index.items():
        if i >= vocab_size:
            continue
        if is_word2vec:
            if word in embeddings_source.wv:
                embedding_matrix[i] = embeddings_source.wv[word]
                found_count += 1
            else:
                embedding_matrix[i] = np.random.normal(0, 0.1, embedding_dim)
        else:
            embedding_vector = embeddings_source.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
                found_count += 1
            else:
                embedding_matrix[i] = np.random.normal(0, 0.1, embedding_dim)

    print(f"Found {found_count}/{vocab_size} words in embeddings.")
    return embedding_matrix

if glove_embeddings_index:
    glove_embedding_matrix = create_embedding_matrix(
        nn_tokenizer.word_index, glove_embeddings_index, EMBEDDING_DIM, vocab_size, is_word2vec=False
    )
else:
    print("Using random embeddings as GloVe is not available.")
    glove_embedding_matrix = np.random.normal(0, 0.1, (vocab_size, EMBEDDING_DIM))

print(f"GloVe Embedding matrix shape: {glove_embedding_matrix.shape}")

In [ ]:
train_texts_tokenized = [text.split() for text in X_train_split]

print("Training Skip-gram (Word2Vec) model...")
skipgram_model = Word2Vec(
    sentences=train_texts_tokenized,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    epochs=10
)

print(f"Skip-gram vocabulary size: {len(skipgram_model.wv)}")
print("Skip-gram model trained successfully!")

In [ ]:
skipgram_embedding_matrix = create_embedding_matrix(
    nn_tokenizer.word_index, skipgram_model, EMBEDDING_DIM, vocab_size, is_word2vec=True
)

print(f"Skip-gram Embedding matrix shape: {skipgram_embedding_matrix.shape}")

## 6. Helper Functions for Evaluation

In [ ]:
results = {}

def evaluate_model(y_true, y_pred, model_name, representation):
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')

    key = f"{model_name}_{representation}"
    results[key] = {
        'Model': model_name,
        'Representation': representation,
        'Accuracy': accuracy,
        'F1_Macro': f1_macro,
        'F1_Weighted': f1_weighted
    }

    print(f"\n{'='*60}")
    print(f"Model: {model_name} with {representation}")
    print(f"{'='*60}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")

    return accuracy, f1_macro, f1_weighted

def plot_confusion_matrix(y_true, y_pred, model_name, representation):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix: {model_name} with {representation}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def print_classification_report(y_true, y_pred, model_name, representation):
    print(f"\n=== Classification Report: {model_name} with {representation} ===")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

print("Evaluation functions defined successfully!")

## 7. Machine Learning Models

In [ ]:
def train_ml_model(model, model_name, X_train, X_val, X_test, y_train, y_val, y_test, representation):
    print(f"\nTraining {model_name} with {representation}...")
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    evaluate_model(y_test, y_test_pred, model_name, representation)
    plot_confusion_matrix(y_test, y_test_pred, model_name, representation)
    print_classification_report(y_test, y_test_pred, model_name, representation)
    return model

ml_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=50, min_samples_split=5, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(alpha=1.0)
}

representations = {
    'BoW': (X_train_bow, X_val_bow, X_test_bow),
    'TF-IDF': (X_train_tfidf, X_val_tfidf, X_test_tfidf)
}

trained_models = {}
for model_name, model in ml_models.items():
    for rep_name, (X_tr, X_v, X_te) in representations.items():
        model_copy = type(model)(**model.get_params())
        trained_models[f"{model_name}_{rep_name}"] = train_ml_model(
            model_copy, model_name, X_tr, X_v, X_te, y_train_split, y_val, y_test, rep_name
        )

## 8. Deep Neural Network with BoW and TF-IDF

In [ ]:
def build_dnn_model(input_dim, num_classes):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("DNN model builder defined successfully!")

In [ ]:
X_train_bow_dense = X_train_bow.toarray()
X_val_bow_dense = X_val_bow.toarray()
X_test_bow_dense = X_test_bow.toarray()

X_train_tfidf_dense = X_train_tfidf.toarray()
X_val_tfidf_dense = X_val_tfidf.toarray()
X_test_tfidf_dense = X_test_tfidf.toarray()

print("Data prepared for DNN training!")

In [ ]:
print("Training DNN with Bag of Words...")
dnn_bow = build_dnn_model(X_train_bow_dense.shape[1], num_classes)
dnn_bow.summary()

history_dnn_bow = dnn_bow.fit(
    X_train_bow_dense, y_train_cat,
    validation_data=(X_val_bow_dense, y_val_cat),
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_dnn_bow.history['accuracy'], label='Train')
axes[0].plot(history_dnn_bow.history['val_accuracy'], label='Validation')
axes[0].set_title('DNN + BoW: Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_dnn_bow.history['loss'], label='Train')
axes[1].plot(history_dnn_bow.history['val_loss'], label='Validation')
axes[1].set_title('DNN + BoW: Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_test_pred_dnn_bow = np.argmax(dnn_bow.predict(X_test_bow_dense), axis=1)
evaluate_model(y_test, y_test_pred_dnn_bow, "DNN", "BoW")
plot_confusion_matrix(y_test, y_test_pred_dnn_bow, "DNN", "BoW")
print_classification_report(y_test, y_test_pred_dnn_bow, "DNN", "BoW")

In [ ]:
print("Training DNN with TF-IDF...")
dnn_tfidf = build_dnn_model(X_train_tfidf_dense.shape[1], num_classes)

history_dnn_tfidf = dnn_tfidf.fit(
    X_train_tfidf_dense, y_train_cat,
    validation_data=(X_val_tfidf_dense, y_val_cat),
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_dnn_tfidf.history['accuracy'], label='Train')
axes[0].plot(history_dnn_tfidf.history['val_accuracy'], label='Validation')
axes[0].set_title('DNN + TF-IDF: Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_dnn_tfidf.history['loss'], label='Train')
axes[1].plot(history_dnn_tfidf.history['val_loss'], label='Validation')
axes[1].set_title('DNN + TF-IDF: Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_test_pred_dnn_tfidf = np.argmax(dnn_tfidf.predict(X_test_tfidf_dense), axis=1)
evaluate_model(y_test, y_test_pred_dnn_tfidf, "DNN", "TF-IDF")
plot_confusion_matrix(y_test, y_test_pred_dnn_tfidf, "DNN", "TF-IDF")
print_classification_report(y_test, y_test_pred_dnn_tfidf, "DNN", "TF-IDF")

## 9. Neural Network Models with GloVe and Skip-gram

In [ ]:
def build_rnn_model(model_type, embedding_matrix, vocab_size, embedding_dim,
                    max_length, num_classes, bidirectional=False):
    model = Sequential()

    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=False
    ))

    if model_type == 'DNN':
        model.add(GlobalAveragePooling1D())
        model.add(Dense(256, activation='relu'))
        model.add(Dropout(0.3))
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(0.3))
    else:
        rnn_units = 128
        dropout = 0.3
        recurrent_dropout = 0.2

        if model_type == 'SimpleRNN':
            rnn_layer = SimpleRNN(rnn_units, dropout=dropout, recurrent_dropout=recurrent_dropout)
        elif model_type == 'GRU':
            rnn_layer = GRU(rnn_units, dropout=dropout, recurrent_dropout=recurrent_dropout)
        elif model_type == 'LSTM':
            rnn_layer = LSTM(rnn_units, dropout=dropout, recurrent_dropout=recurrent_dropout)

        if bidirectional:
            model.add(Bidirectional(rnn_layer))
        else:
            model.add(rnn_layer)

        model.add(Dropout(0.3))
        model.add(Dense(64, activation='relu'))
        model.add(Dropout(0.2))

    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

print("RNN model builder defined successfully!")

In [ ]:
def train_and_evaluate_nn(model_type, embedding_matrix, embedding_name,
                          bidirectional=False, epochs=10, batch_size=128):
    model_name = f"Bidirectional {model_type}" if bidirectional else model_type
    print(f"\n{'='*60}")
    print(f"Training {model_name} with {embedding_name}...")
    print(f"{'='*60}")

    model = build_rnn_model(
        model_type=model_type,
        embedding_matrix=embedding_matrix,
        vocab_size=vocab_size,
        embedding_dim=EMBEDDING_DIM,
        max_length=MAX_SEQUENCE_LENGTH,
        num_classes=num_classes,
        bidirectional=bidirectional
    )

    model.summary()

    history = model.fit(
        X_train_pad, y_train_cat,
        validation_data=(X_val_pad, y_val_cat),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=1
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{model_name} + {embedding_name}: Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{model_name} + {embedding_name}: Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    y_test_pred = np.argmax(model.predict(X_test_pad), axis=1)
    evaluate_model(y_test, y_test_pred, model_name, embedding_name)
    plot_confusion_matrix(y_test, y_test_pred, model_name, embedding_name)
    print_classification_report(y_test, y_test_pred, model_name, embedding_name)

    return model, history

print("Training function defined successfully!")

In [ ]:
dnn_glove_model, dnn_glove_history = train_and_evaluate_nn(
    model_type='DNN',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    epochs=15
)

In [ ]:
dnn_skipgram_model, dnn_skipgram_history = train_and_evaluate_nn(
    model_type='DNN',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    epochs=15
)

In [ ]:
simplernn_glove_model, simplernn_glove_history = train_and_evaluate_nn(
    model_type='SimpleRNN',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    epochs=15
)

In [ ]:
simplernn_skipgram_model, simplernn_skipgram_history = train_and_evaluate_nn(
    model_type='SimpleRNN',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    epochs=15
)

In [ ]:
gru_glove_model, gru_glove_history = train_and_evaluate_nn(
    model_type='GRU',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    epochs=15
)

In [ ]:
gru_skipgram_model, gru_skipgram_history = train_and_evaluate_nn(
    model_type='GRU',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    epochs=15
)

In [ ]:
lstm_glove_model, lstm_glove_history = train_and_evaluate_nn(
    model_type='LSTM',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    epochs=15
)

In [ ]:
lstm_skipgram_model, lstm_skipgram_history = train_and_evaluate_nn(
    model_type='LSTM',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    epochs=15
)

In [ ]:
bi_simplernn_glove_model, bi_simplernn_glove_history = train_and_evaluate_nn(
    model_type='SimpleRNN',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    bidirectional=True,
    epochs=15
)

In [ ]:
bi_simplernn_skipgram_model, bi_simplernn_skipgram_history = train_and_evaluate_nn(
    model_type='SimpleRNN',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    bidirectional=True,
    epochs=15
)

In [ ]:
bi_gru_glove_model, bi_gru_glove_history = train_and_evaluate_nn(
    model_type='GRU',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    bidirectional=True,
    epochs=15
)

In [ ]:
bi_gru_skipgram_model, bi_gru_skipgram_history = train_and_evaluate_nn(
    model_type='GRU',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    bidirectional=True,
    epochs=15
)

In [ ]:
bi_lstm_glove_model, bi_lstm_glove_history = train_and_evaluate_nn(
    model_type='LSTM',
    embedding_matrix=glove_embedding_matrix,
    embedding_name='GloVe',
    bidirectional=True,
    epochs=15
)

In [ ]:
bi_lstm_skipgram_model, bi_lstm_skipgram_history = train_and_evaluate_nn(
    model_type='LSTM',
    embedding_matrix=skipgram_embedding_matrix,
    embedding_name='Skip-gram',
    bidirectional=True,
    epochs=15
)

## 10. Results Summary and Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.reset_index(drop=True)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("=" * 80)
print("COMPLETE RESULTS SUMMARY")
print("=" * 80)
print(results_df.to_string())

In [ ]:
plt.figure(figsize=(16, 10))

results_df['Label'] = results_df['Model'] + ' + ' + results_df['Representation']

x = range(len(results_df))
width = 0.25

plt.barh([i - width for i in x], results_df['Accuracy'], width, label='Accuracy', color='steelblue')
plt.barh([i for i in x], results_df['F1_Macro'], width, label='F1 (Macro)', color='coral')
plt.barh([i + width for i in x], results_df['F1_Weighted'], width, label='F1 (Weighted)', color='seagreen')

plt.yticks(x, results_df['Label'])
plt.xlabel('Score')
plt.title('Model Performance Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
ml_models = ['Random Forest', 'Logistic Regression', 'Naive Bayes']
nn_models = ['DNN', 'SimpleRNN', 'GRU', 'LSTM',
             'Bidirectional SimpleRNN', 'Bidirectional GRU', 'Bidirectional LSTM']

ml_results = results_df[results_df['Model'].isin(ml_models)].copy()
nn_results = results_df[results_df['Model'].isin(nn_models)].copy()

print("\n" + "="*60)
print("MACHINE LEARNING MODELS RESULTS")
print("="*60)
print(ml_results[['Model', 'Representation', 'Accuracy', 'F1_Macro', 'F1_Weighted']].to_string(index=False))

print("\n" + "="*60)
print("NEURAL NETWORK MODELS RESULTS")
print("="*60)
print(nn_results[['Model', 'Representation', 'Accuracy', 'F1_Macro', 'F1_Weighted']].to_string(index=False))

In [ ]:
print("\n" + "="*60)
print("BEST AND WORST PERFORMING MODELS")
print("="*60)

if len(ml_results) > 0:
    best_ml = ml_results.loc[ml_results['Accuracy'].idxmax()]
    worst_ml = ml_results.loc[ml_results['Accuracy'].idxmin()]

    print("\n--- Best ML Model ---")
    print(f"Model: {best_ml['Model']} with {best_ml['Representation']}")
    print(f"Accuracy: {best_ml['Accuracy']:.4f}")
    print(f"F1 Macro: {best_ml['F1_Macro']:.4f}")
    print(f"F1 Weighted: {best_ml['F1_Weighted']:.4f}")

    print("\n--- Worst ML Model ---")
    print(f"Model: {worst_ml['Model']} with {worst_ml['Representation']}")
    print(f"Accuracy: {worst_ml['Accuracy']:.4f}")
    print(f"F1 Macro: {worst_ml['F1_Macro']:.4f}")
    print(f"F1 Weighted: {worst_ml['F1_Weighted']:.4f}")

if len(nn_results) > 0:
    best_nn = nn_results.loc[nn_results['Accuracy'].idxmax()]
    worst_nn = nn_results.loc[nn_results['Accuracy'].idxmin()]

    print("\n--- Best NN Model ---")
    print(f"Model: {best_nn['Model']} with {best_nn['Representation']}")
    print(f"Accuracy: {best_nn['Accuracy']:.4f}")
    print(f"F1 Macro: {best_nn['F1_Macro']:.4f}")
    print(f"F1 Weighted: {best_nn['F1_Weighted']:.4f}")

    print("\n--- Worst NN Model ---")
    print(f"Model: {worst_nn['Model']} with {worst_nn['Representation']}")
    print(f"Accuracy: {worst_nn['Accuracy']:.4f}")
    print(f"F1 Macro: {worst_nn['F1_Macro']:.4f}")
    print(f"F1 Weighted: {worst_nn['F1_Weighted']:.4f}")

In [ ]:
print("\n" + "="*60)
print("COMPARISON: BEST ML vs BEST NN")
print("="*60)

if len(ml_results) > 0 and len(nn_results) > 0:
    comparison_data = {
        'Metric': ['Model', 'Representation', 'Accuracy', 'F1 Macro', 'F1 Weighted'],
        'Best ML': [
            best_ml['Model'],
            best_ml['Representation'],
            f"{best_ml['Accuracy']:.4f}",
            f"{best_ml['F1_Macro']:.4f}",
            f"{best_ml['F1_Weighted']:.4f}"
        ],
        'Best NN': [
            best_nn['Model'],
            best_nn['Representation'],
            f"{best_nn['Accuracy']:.4f}",
            f"{best_nn['F1_Macro']:.4f}",
            f"{best_nn['F1_Weighted']:.4f}"
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 8))

pivot_accuracy = results_df.pivot_table(
    values='Accuracy',
    index='Model',
    columns='Representation',
    aggfunc='mean'
)

sns.heatmap(pivot_accuracy, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.5, cbar_kws={'label': 'Accuracy'})
plt.title('Model Accuracy by Representation Type')
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*60)
print("REPRESENTATION COMPARISON")
print("="*60)

rep_comparison = results_df.groupby('Representation').agg({
    'Accuracy': ['mean', 'std', 'max'],
    'F1_Macro': ['mean', 'std', 'max'],
    'F1_Weighted': ['mean', 'std', 'max']
}).round(4)

print(rep_comparison)

In [ ]:
results_df.to_csv('experiment_results.csv', index=False)
print("Results saved to 'experiment_results.csv'")